<a href="https://colab.research.google.com/github/Rapsim/IPEO_DeepL_group15/blob/raph/explo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### code adapté de l'exo 9

In [ ]:
import os
import torch
from torch.utils.data import dataset
from torch.utils.data import DataLoader
import numpy as np
import tifffile


In [ ]:
# Number of classes: 0 = nodata to 37 = last class in CSV ?
NUM_CLASSES = 38


### dataset


In [ ]:


class VaihingenDataset(dataset.Dataset):
    """
    Custom Dataset class that loads Sentinel-2 image patches and
    land cover masks from your Amazon dataset.

    We keep the old name 'VaihingenDataset' so the rest of the
    notebook works without big changes.
    """

    def __init__(self, data_root, split="train"):
        """
        data_root: folder that contains the subfolders:
            data_root/S2/*.tif
            data_root/labels/*.tif

        split: "train", "val", or "test"
        """
        super().__init__()
        self.data_root = data_root
        self.split = split

        self.s2_dir = os.path.join(data_root, "S2")
        self.label_dir = os.path.join(data_root, "labels")

        # all filenames in S2 folder
        all_fnames = sorted(
            [f for f in os.listdir(self.s2_dir) if f.endswith(".tif")]
        )

        # separate train and test by prefix
        train_fnames = [f for f in all_fnames if f.startswith("train")]
        test_fnames  = [f for f in all_fnames if f.startswith("test")]

        if split in ("train", "val"):
            # deterministic 80/20 split of the train_ files
            n_train = int(0.8 * len(train_fnames))
            train_fnames = sorted(train_fnames)
            if split == "train":
                selected = train_fnames[:n_train]
            else:  # "val"
                selected = train_fnames[n_train:]
        elif split == "test":
            selected = sorted(test_fnames)
        else:
            raise ValueError(f"Unknown split: {split}")

        self.fnames = selected

        print(f"{split} split has {len(self.fnames)} samples.")

    def __len__(self):
        return len(self.fnames)

    def _read_tiff_image(self, path):
        """
        Read a .tif file as a numpy array and return it as (C, H, W).

        Works whether data is stored as (H, W, C) or (C, H, W) or (H, W).
        """
        arr = tifffile.imread(path)  # numpy array

        if arr.ndim == 2:
            # single band -> (1, H, W)
            arr = arr[np.newaxis, ...]
        elif arr.ndim == 3:
            # guess if channels are first or last
            if arr.shape[0] in (1, 3, 4, 12, 64):
                # assume (C, H, W)
                pass
            elif arr.shape[-1] in (1, 3, 4, 12, 64):
                # assume (H, W, C)
                arr = np.transpose(arr, (2, 0, 1))
            else:
                raise ValueError(f"Unexpected image shape {arr.shape} for {path}")
        else:
            raise ValueError(f"Unexpected ndim {arr.ndim} for {path}")

        return arr

    def __getitem__(self, idx):
        fname = self.fnames[idx]

        # Sentinel-2 image path and label path
        img_path = os.path.join(self.s2_dir, fname)
        label_path = os.path.join(self.label_dir, fname)

        # ---- Read Sentinel-2 patch ----
        img = self._read_tiff_image(img_path).astype(np.float32)  # (C, H, W)

        # Sentinel-2 typical values up to ~10000 -> scale to ~[0, 1]
        img = img / 10000.0

        # ---- Read label mask ----
        mask = tifffile.imread(label_path)  # often (H, W) or (1, H, W)

        if mask.ndim == 3:
            # squeeze if there is a singleton channel dimension
            if mask.shape[0] == 1:
                mask = mask[0]
            elif mask.shape[-1] == 1:
                mask = mask[..., 0]
            else:
                raise ValueError(f"Unexpected mask shape {mask.shape} for {label_path}")

        if mask.ndim != 2:
            raise ValueError(f"Mask must be 2D, got shape {mask.shape}")

        mask = mask.astype(np.int64)  # required for CrossEntropyLoss

        # ---- Convert to torch tensors ----
        inputs = torch.from_numpy(img)    # (C, H, W), C should be 12
        labels = torch.from_numpy(mask)   # (H, W), ints in [0..62] (0 = nodata)

        return inputs, labels


def load_dataloader(batch_size, split="train"):
    """
    split in {"train", "val", "test"}.

    The global variable `data_root` should point to the folder that
    contains the S2/ and labels/ folders.
    """
    return DataLoader(
        VaihingenDataset(data_root, split=split),
        batch_size=batch_size,
        shuffle=(split == "train"),
        num_workers=2,
    )
